# Evidencias de Reproduccion Experimental - Lab 02 Part 02

Este notebook maestro resume el estado general de la reproduccion experimental.

La evidencia detallada ya esta repartida en los notebooks semanales:
- `semana-01-datos.ipynb`
- `semana-02-m2-sin-vacios.ipynb`
- `semana-03-m2-con-vacios.ipynb`
- `semana-04-cierre.ipynb`

Este archivo queda como panel global para mostrar:
- estructura actual de `Part-02`,
- entorno y versiones,
- artefactos generados,
- metricas resumidas,
- y comparacion final con el paper.

In [ ]:
from pathlib import Path
import json
import platform
import sys
from importlib.metadata import version
import pandas as pd

cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]

PART02 = next((p for p in search_roots if p.name == 'Part-02' and p.parent.name == 'Lab-02'), None)
if PART02 is None:
    PART02 = next((p / 'laboratorios' / 'Lab-02' / 'Part-02' for p in search_roots if (p / 'laboratorios' / 'Lab-02' / 'Part-02').exists()), None)
if PART02 is None:
    raise FileNotFoundError('No se pudo ubicar la carpeta laboratorios/Lab-02/Part-02 desde el directorio actual.')

ROOT = PART02.parents[2]
DOCS = PART02 / 'docs'
EVIDENCIAS = PART02 / 'evidencias'
RESULTADOS = PART02 / 'resultados'
EXPERIMENTOS = RESULTADOS / 'experimentos' / 'M2'
METRICAS = RESULTADOS / 'comparativas' / 'metricas' / 'M2'
NOTEBOOKS = PART02 / 'notebooks'

def relpath(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

print('ROOT        :', ROOT)
print('PART02      :', PART02)
print('DOCS        :', DOCS)
print('EVIDENCIAS  :', EVIDENCIAS)
print('RESULTADOS  :', RESULTADOS)
print('EXPERIMENTOS:', EXPERIMENTOS)
print('METRICAS    :', METRICAS)

## Entorno y versiones

In [ ]:
print('Python executable:', sys.executable)
print('Python version   :', sys.version)
print('Platform         :', platform.platform())

for package_name, label in [
    ('tensorflow', 'tensorflow'),
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('scipy', 'scipy'),
    ('scikit-learn', 'scikit-learn'),
]:
    print(f'{label:<17}:', version(package_name))

## Estructura clave del proyecto

In [ ]:
paths_to_show = [
    PART02 / 'src' / 'prepare_datasets.py',
    PART02 / 'src' / 'train_m2.py',
    PART02 / 'src' / 'evaluate_m2.py',
    PART02 / 'docs' / 'PLAN-4-SEMANAS.md',
    PART02 / 'docs' / 'cronograma' / 'semana-01.md',
    PART02 / 'docs' / 'cronograma' / 'semana-02.md',
    PART02 / 'docs' / 'cronograma' / 'semana-03.md',
    PART02 / 'docs' / 'cronograma' / 'semana-04.md',
    PART02 / 'evidencias' / 'base' / 'comandos-ejecutados.txt',
    PART02 / 'evidencias' / 'base' / 'metricas-reproduccion.csv',
    PART02 / 'resultados' / 'experimentos' / 'M2' / 'base' / 'with_empty_values' / 'model.keras',
    PART02 / 'resultados' / 'experimentos' / 'M2' / 'base' / 'without_empty_values' / 'model.keras',
    PART02 / 'resultados' / 'comparativas' / 'metricas' / 'M2' / 'base' / 'with_empty_values_evaluation.json',
    PART02 / 'resultados' / 'comparativas' / 'metricas' / 'M2' / 'base' / 'without_empty_values_evaluation.json',
]

pd.DataFrame([
    {'ruta': relpath(path), 'exists': path.exists()} for path in paths_to_show
])

## Tabla resumen de metricas

In [ ]:
metrics_df = pd.read_csv(EVIDENCIAS / 'base' / 'metricas-reproduccion.csv')
metrics_df

## Comparacion rapida con el paper

In [ ]:
paper = {
    'with_empty_values': {'mean': 2.0368, 'median': 1.8199},
    'with_empty_values_seed21': {'mean': 2.0368, 'median': 1.8199},
    'without_empty_values': {'mean': 2.0559, 'median': 1.7745},
}

rows = []
for _, row in metrics_df.iterrows():
    ref = paper[row['escenario']]
    rows.append({
        'escenario': row['escenario'],
        'paper_mean': ref['mean'],
        'reproduction_mean': row['mean_m'],
        'delta_mean': round(row['mean_m'] - ref['mean'], 4),
        'paper_median': ref['median'],
        'reproduction_median': row['median_m'],
        'delta_median': round(row['median_m'] - ref['median'], 4),
    })

comparison_df = pd.DataFrame(rows)
comparison_df

## Archivos de evaluacion disponibles

In [ ]:
files = []
for folder in [EXPERIMENTOS, METRICAS, EVIDENCIAS / 'base', NOTEBOOKS]:
    for path in sorted(folder.rglob('*')):
        if path.is_file():
            files.append({
                'ruta': relpath(path),
                'tamano_bytes': path.stat().st_size,
            })

files_df = pd.DataFrame(files)
files_df.head(40)

## Rutas de los notebooks semanales

In [ ]:
weekly = [
    NOTEBOOKS / 'semana-01-datos.ipynb',
    NOTEBOOKS / 'semana-02-m2-sin-vacios.ipynb',
    NOTEBOOKS / 'semana-03-m2-con-vacios.ipynb',
    NOTEBOOKS / 'semana-04-cierre.ipynb',
]
pd.DataFrame([{'notebook': relpath(p), 'exists': p.exists()} for p in weekly])

## Conclusiones para capturas

Las mejores salidas para mostrar rapido son:
1. entorno y versiones,
2. tabla de metricas,
3. comparacion con el paper,
4. verificacion de rutas clave,
5. listado de notebooks semanales.